## Install Libraries

In [49]:
!pip install pandas numpy sqlalchemy pymysql

Defaulting to user installation because normal site-packages is not writeable


In [50]:
from sqlalchemy import create_engine, text
import pandas as pd
import numpy as np

## Load and Explore the Dataset

In [11]:
df= pd.read_csv(r"C:\Users\JANVIPRIYANSHI\Downloads\flipkart_dataset.csv")

In [14]:
df.head()

,order_id,payment_id,user_id,product_id,category,sub_category,timestamp,price,quantity,discount_pct,total_amount,payment_method,order_status,is_fraud
0,1,1,2119,2867,Fashion,Clothing & Accessories,2022-07-28 15:46:39,411.17,1,16.6,342.92,UPI,Completed,0
1,2,2,1299,2782,Fashion,Clothing & Accessories,2022-05-25 15:46:06,154.89,1,8.8,141.26,Net Banking,Cancelled,0
2,3,3,1556,2376,Appliances,Large Appliances,2022-02-27 12:24:27,714.16,1,15.1,606.32,UPI,Completed,0
3,4,4,1459,2127,Electronics,Mobiles & Laptops,2022-07-10 16:02:05,579.54,2,14.9,986.38,UPI,Completed,0
4,5,5,1683,2754,Fashion,Clothing & Accessories,2022-09-10 23:27:12,259.20,2,6.7,483.67,Wallet,Cancelled,0


In [16]:
df.shape

(12000, 14)

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        12000 non-null  int64  
 1   payment_id      12000 non-null  int64  
 2   user_id         12000 non-null  int64  
 3   product_id      12000 non-null  int64  
 4   category        12000 non-null  object 
 5   sub_category    12000 non-null  object 
 6   timestamp       12000 non-null  object 
 7   price           12000 non-null  float64
 8   quantity        12000 non-null  int64  
 9   discount_pct    12000 non-null  float64
 10  total_amount    12000 non-null  float64
 11  payment_method  12000 non-null  object 
 12  order_status    12000 non-null  object 
 13  is_fraud        12000 non-null  int64  
dtypes: float64(3), int64(6), object(5)
memory usage: 1.3+ MB


## Data Cleaning

In [23]:
# Convert timestamp to datetime
df['timestamp']=pd.to_datetime(df['timestamp'])

In [25]:
# Extract time features
df['order_year']=df['timestamp'].dt.year

In [27]:
df['order_month']=df['timestamp'].dt.month

In [29]:
df['order_month_name']=df['timestamp'].dt.strftime('%B')

In [31]:
df['order_quarter']=df['timestamp'].dt.quarter

In [33]:
df['order_hour']=df['timestamp'].dt.hour

In [35]:
df['day_of_week']=df['timestamp'].dt.day_name()

In [37]:
# Add Average Order Value feature
df['aov']=(df['total_amount']/df['quantity']).round(2)

In [41]:
# Confirm no nulls after cleaning
df.isnull().sum().sum()

0

In [43]:
df.shape

(12000, 21)

In [45]:
df.head()

,order_id,payment_id,user_id,product_id,category,sub_category,timestamp,price,quantity,discount_pct,...,payment_method,order_status,is_fraud,order_year,order_month,order_month_name,order_quarter,order_hour,day_of_week,aov
0,1,1,2119,2867,Fashion,Clothing & Accessories,2022-07-28 15:46:39,411.17,1,16.6,...,UPI,Completed,0,2022,7,July,3,15,Thursday,342.92
1,2,2,1299,2782,Fashion,Clothing & Accessories,2022-05-25 15:46:06,154.89,1,8.8,...,Net Banking,Cancelled,0,2022,5,May,2,15,Wednesday,141.26
2,3,3,1556,2376,Appliances,Large Appliances,2022-02-27 12:24:27,714.16,1,15.1,...,UPI,Completed,0,2022,2,February,1,12,Sunday,606.32
3,4,4,1459,2127,Electronics,Mobiles & Laptops,2022-07-10 16:02:05,579.54,2,14.9,...,UPI,Completed,0,2022,7,July,3,16,Sunday,493.19
4,5,5,1683,2754,Fashion,Clothing & Accessories,2022-09-10 23:27:12,259.20,2,6.7,...,Wallet,Cancelled,0,2022,9,September,3,23,Saturday,241.84


## Recency, Frequency, Monetary(RFM)

In [59]:
rfm= df.groupby('user_id').agg(
    total_orders=('order_id', 'count'),
    lifetime_value=('total_amount', 'sum')
).reset_index()

In [61]:
rfm['lifetime_value'] = rfm['lifetime_value'].round(2)

In [63]:
def segment_user(row):
    if row['total_orders'] >= 8 and row['lifetime_value'] >= 3000:
        return 'Loyal'
    elif row['total_orders'] >= 4:
        return 'Regular'
    else:
        return 'New'


In [65]:
rfm['user_segment'] = rfm.apply(segment_user, axis=1)

In [67]:
print('Segment distribution:')
print(rfm['user_segment'].value_counts())

Segment distribution:
user_segment
Loyal      599
Regular    340
New        251
Name: count, dtype: int64


In [70]:
rfm

,user_id,total_orders,lifetime_value,user_segment
0,1001,8,6100.49,Loyal
1,1002,3,889.81,New
2,1003,8,6706.08,Loyal
3,1004,14,9812.42,Loyal
4,1005,4,1883.81,Regular
...,...,...,...,...
1185,2196,12,7456.42,Loyal
1186,2197,13,10655.84,Loyal
1187,2198,13,8909.27,Loyal
1188,2199,22,27058.87,Loyal


## Split Into 4 Database Tables

In [72]:
# TABLE 1 — dim_users
# One row per unique user with their RFM segment
dim_users = rfm[['user_id','user_segment','total_orders','lifetime_value']].copy()
print('dim_users     :', dim_users.shape)     


dim_users     : (1190, 4)


In [74]:
dim_products= df[['product_id','category','sub_category','price']].drop_duplicates(subset='product_id').reset_index(drop=True)

In [78]:
print('dim_products  :', dim_products.shape)

dim_products  : (900, 4)


In [80]:
# TABLE 3 — fact_orders
# Every transaction — core fact table
fact_orders = df[[
    'order_id','user_id','product_id','timestamp',
    'order_year','order_month','order_month_name',
    'order_quarter','order_hour','day_of_week',
    'quantity','discount_pct','total_amount','order_status','aov'
]].copy()
print('fact_orders   :', fact_orders.shape)   


fact_orders   : (12000, 15)


In [82]:
# TABLE 4 — fact_payments
# Payment detail per order
fact_payments = df[['payment_id','order_id','payment_method','is_fraud']].copy()
print('fact_payments :', fact_payments.shape)


fact_payments : (12000, 4)


## Connect Jupyter to MySQL

In [96]:
# ── Enter your MySQL credentials ──────────────────────────
MYSQL_USER     = 'root'
MYSQL_PASSWORD = 'your_password'   # your MySQL Workbench password
MYSQL_HOST     = 'localhost'
MYSQL_PORT     = '3306'
MYSQL_DB       = 'flipkart_db'
 
engine = create_engine(
    f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}'
)
 
# Test connection
with engine.connect() as conn:
    db = conn.execute(text('SELECT DATABASE()')).fetchone()[0]
    print(f'Connected to: {db}')

Connected to: flipkart_db


## Push All 4 Tables Directly Into MySQL

In [99]:
 
tables = {
    'dim_users'    : dim_users,
    'dim_products' : dim_products,
    'fact_orders'  : fact_orders,
    'fact_payments': fact_payments,
}
 
for name, dataframe in tables.items():
    dataframe.to_sql(
        name      = name,
        con       = engine,
        if_exists = 'replace',   # drops & recreates table fresh each run
        index     = False,       # do not write pandas row index
        chunksize = 500          # send 500 rows at a time
    )
    print(f'  Pushed {name:20s} — {len(dataframe):,} rows')
 
print()
print('All tables sent to MySQL successfully!')


  Pushed dim_users            — 1,190 rows
  Pushed dim_products         — 900 rows
  Pushed fact_orders          — 12,000 rows
  Pushed fact_payments        — 12,000 rows

All tables sent to MySQL successfully!


## Verify Tables Reached MySQL

In [102]:
with engine.connect() as conn:
    for t in ['dim_users','dim_products','fact_orders','fact_payments']:
        count = conn.execute(text(f'SELECT COUNT(*) FROM {t}')).fetchone()[0]
        print(f'{t:25s}  →  {count:,} rows')


dim_users                  →  1,190 rows
dim_products               →  900 rows
fact_orders                →  12,000 rows
fact_payments              →  12,000 rows
